In [1]:
import numpy as np
import asyncio
from bqplot import LinearScale, Scatter, Lines, Figure
import ipywidgets as widgets
from IPython.display import display

# 1. State
state = {
    "x_cart": 0.0,
    "x_target": 0.0,  # This is where your mouse is
    "x_velocity": 0.0,
    "theta": 0.5,
    "w": 0.0,
    "l": 1.0,
    "g": 9.81,
    "dt": 0.02,
}

# 2. Setup Plot
sc_x = LinearScale(min=-2.5, max=2.5)
sc_y = LinearScale(min=-1.5, max=1.5)

cart = Scatter(
    x=[state["x_cart"]],
    y=[0],
    scales={"x": sc_x, "y": sc_y},
    colors=["white"],
    marker="square",
    default_size=1000,
    enable_move=True,
    continuous_update=True,
)


def get_pole_coords():
    px = state["x_cart"] + state["l"] * np.sin(state["theta"])
    py = -state["l"] * np.cos(state["theta"])
    return [state["x_cart"], px], [0, py]


lx, ly = get_pole_coords()
pole = Lines(x=lx, y=ly, scales={"x": sc_x, "y": sc_y}, colors=["cyan"], stroke_width=4)
tip = Scatter(
    x=[lx[1]],
    y=[ly[1]],
    scales={"x": sc_x, "y": sc_y},
    colors=["magenta"],
    default_size=50,
)


# 3. Handle Mouse Input
def on_cart_move(change):
    # We update a 'TARGET', but we let the physics loop
    # move the actual cart so the pole stays attached
    state["x_target"] = change["new"][0]


cart.observe(on_cart_move, names=["x"])


# 4. Sync Function
def update_plot():
    lx, ly = get_pole_coords()
    with fig.hold_sync():
        cart.x = [state["x_cart"]]
        pole.x, pole.y = lx, ly
        tip.x, tip.y = [lx[1]], [ly[1]]


# 5. The Unified Physics Loop
async def physics_loop():
    while True:
        # Calculate how fast the cart is being moved by your hand
        dx = state["x_target"] - state["x_cart"]
        cart_vel = dx / state["dt"]
        cart_accel = (cart_vel - state["x_velocity"]) / state["dt"]

        # Update cart state
        state["x_cart"] = state["x_target"]
        state["x_velocity"] = cart_vel

        # Pendulum Physics: alpha = gravity_term + inertial_term
        # The (cart_accel * cos) term is the force from your hand
        alpha = (-(state["g"] / state["l"]) * np.sin(state["theta"])) - (
            (cart_accel / state["l"]) * np.cos(state["theta"])
        )

        state["w"] += alpha * state["dt"]
        state["w"] *= 0.98  # Friction
        state["theta"] += state["w"] * state["dt"]

        update_plot()
        await asyncio.sleep(state["dt"])


# 6. Figure
fig = Figure(
    marks=[pole, cart, tip],
    title="Momentum Transfer: Flick the cart to see the swing",
    background_style={"fill": "#121212"},
    animation_duration=0,
)

display(fig)

loop = asyncio.get_event_loop()
loop.create_task(physics_loop())

ModuleNotFoundError: No module named 'bqplot'